In [ ]:
#@title # Environment Setup
#@markdown Install vedo and setup virtual rendering for Colab.

!pip install -q vedo
!apt-get install -q -y xvfb freeglut3-dev ffmpeg > /dev/null

import os
os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

from vedo import *
# This tells vedo to show the plots directly in the Colab cell
settings.default_backend = 'k3d'


#@title # 1. Mathematical Constants & Soft-Min
#@markdown This section handles the logic for selecting the "best" exit smoothly.

import numpy as np
from vedo import *

H = 3.0           # Height of each floor
N_AGENTS = 50     # Start with 50, scale up later
ALPHA = 0.05      # Learning rate (step size)
K_SOFT = 2.0      # Soft-min "sharpness"

# Exits on Ground Floor (z=0)
EXIT_A = np.array([10, 0, 0])
EXIT_B = np.array([-10, 0, 0])

def soft_min_cost(pos, targets, k=K_SOFT):
    """Calculates a smooth cost field between multiple exits."""
    # Cost to each target (distance squared)
    costs = [0.5 * np.sum((pos - t)**2) for t in targets]
    # Soft-min formula: -1/k * ln(sum(exp(-k * cost)))
    return - (1.0/k) * np.log(np.sum(np.exp(-k * np.array(costs))))

def get_soft_min_gradient(pos, targets, k=K_SOFT):
    """Calculates the gradient pull toward the 'soft' best exit."""
    # Numerical gradient approximation is often most stable for soft-min
    eps = 1e-4
    grad = np.zeros(3)
    for i in range(3):
        p_plus = pos.copy()
        p_minus = pos.copy()
        p_plus[i] += eps
        p_minus[i] -= eps
        grad[i] = (soft_min_cost(p_plus, targets, k) - soft_min_cost(p_minus, targets, k)) / (2 * eps)
    return grad


#@title # 2. 3D Environment Rendering
#@markdown Creates the floor planes, ramps, and 3D walls.

def build_environment():
    objs = []
    # Floors
    for i in range(3):
        z_level = i * H
        objs.append(Plane(pos=(0,0,z_level), normal=(0,0,1), s=(25,25), c="white", alpha=0.2))

    # Example Ramp: From 2nd Floor (z=6) to 1st Floor (z=3)
    # Ramps are defined by a start point, end point, and width
    ramp1 = Mesh([[0,5,6], [10,5,3], [10,8,3], [0,8,6]], c="blue7").alpha(0.5)
    objs.append(ramp1)

    # Boundary Walls (High)
    objs.append(Box(pos=(0, 12, 4.5), length=25, width=0.2, height=9, c="gray5"))

    # Internal Walls (Low - as requested)
    objs.append(Box(pos=(0, 0, 0.5), length=1, width=8, height=1, c="gray8"))

    return objs


#@title # 3. Ramp and Floor Gradient Logic
#@markdown The math that keeps particles from falling through ceilings.

def get_ramp_gradient(pos, r_start, r_end, r_width):
    """Provides a 'gravity' pull along the ramp and keeps agents centered."""
    # Vector along the ramp
    ramp_vec = r_end - r_start
    ramp_len = np.linalg.norm(ramp_vec)
    u = ramp_vec / ramp_len

    # Projection of agent onto ramp line
    rel_pos = pos - r_start
    dist_along = np.dot(rel_pos, u)

    if 0 <= dist_along <= ramp_len:
        # 1. Pull toward the bottom of the ramp
        grad_down = -u
        # 2. Centering force (keep them from falling off the sides)
        projection = r_start + dist_along * u
        dist_side = pos - projection
        grad_center = dist_side
        return grad_down + 5.0 * grad_center
    return np.zeros(3)



